In [1]:
import polars as pl
from pybiomart import Server
import os
import numpy as np
import pandas as pd

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/dataset'
HISTONE_DIR = os.path.join(WORKING_DIR, 'dataset', 'histone_overlap')

In [3]:
# Schema for gene data
schema_genes = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64()
})

In [7]:
# Read the gene data
gene_pl = pl.read_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_top1.csv"),
                     schema=schema_genes,
                     has_header=False,
                     separator="\t")

In [8]:
gene_pl

chromosome,tss -2kb,tss +2kb,test_id,gene_id,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,locus,orig_start,orig_end
str,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,str,i64,i64
"""chr1""",67091,71091,"""XLOC_000001""","""XLOC_000001""","""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,"""chr1:69090-70008""",69090,70008
"""chr1""",365640,369640,"""XLOC_000003""","""XLOC_000003""","""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,"""chr1:367658-368597""",367658,368597
"""chr1""",858260,862260,"""XLOC_000006""","""XLOC_000006""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473
"""chr1""",858260,862260,"""XLOC_000007""","""XLOC_000007""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",24327129,24331129,"""XLOC_030009""","""XLOC_030009""","""Y""",24314689,24329129,-1,"""RBMY1F""","""ENSG00000169800""","""protein_coding""",24329129,"""chrY:24314688-24329089""",24314688,24329089
"""chrY""",25343241,25347241,"""XLOC_030012""","""XLOC_030012""","""Y""",25275502,25345241,-1,"""DAZ1""","""ENSG00000188120""","""protein_coding""",25345241,"""chrY:25275501-25345239""",25275501,25345239
"""chrY""",26192166,26196166,"""XLOC_030014""","""XLOC_030014""","""Y""",26191376,26194166,-1,"""CDY1B""","""ENSG00000172352""","""protein_coding""",26194166,"""chrY:26191376-26194161""",26191376,26194161


In [9]:
gene_pl.group_by("chromosome").len().sort("len", descending=True)

chromosome,len
str,u32
"""chr1""",2262
"""chr19""",1468
"""chr2""",1423
"""chr11""",1384
"""chr17""",1288
…,…
"""chr22""",505
"""chr13""",364
"""chr18""",314


In [10]:
gene_list = gene_pl.select(pl.col('gene_id')).to_numpy().flatten()
gene_list

array(['XLOC_000001', 'XLOC_000003', 'XLOC_000006', ..., 'XLOC_030014',
       'XLOC_030017', 'XLOC_030018'], dtype=object)

In [11]:
# Schema for overlap histone
schema_histone_overlap = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64(),
    "histone_chr"               : pl.String(),
    "histone_start"             : pl.Int64(),
    "histone_end"               : pl.Int64(),
    "histone_name"              : pl.String()
})

In [ ]:
histone_names = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

In [ ]:
def build_matrix(gene_list, histone_name):
    # Open histone overlap file
    histone_pl = pl.read_csv(os.path.join(HISTONE_DIR, f'ensembl_top1_{histone_name}.bed'), 
                         schema = schema_histone_overlap, 
                         separator="\t",
                         has_header=False)

    # Create partitions to optimize searching
    histone_partitions = histone_pl.partition_by("gene_id", as_dict=True)

    # Build histone matrix
    histone_list = []
    i = 1
    for gene in gene_list:
        histone_arr = np.zeros(4000, dtype=int)
        count = 0
        
        if((gene,) in histone_partitions):
            # print(f"{gene} -  EXIST")
            
            partition = histone_partitions[(gene,)].to_numpy()
            count = partition.shape[0]
    
            for row in partition:
                start = row[1]
                idx_start = row[17] - start
                idx_end = row[18] - start
                # print(f"{row[19]}: {idx_start} - {idx_end}")
                
                histone_arr[idx_start: idx_end] = 1
        # else:
        #     print(f"{gene} -  NOT EXIST")
    
        # print(f"SUM: {np.sum(histone_arr)}, COUNT: {count}")
        entry = [gene, histone_arr, count]
    
        # print(f"{i}/{len(gene_list)}: {gene}, count: {count}")
    
        histone_list.append(entry)
        i += 1

    

In [12]:
histone_pl = pl.read_csv(os.path.join(HISTONE_DIR, 'ensembl_top1_H3K4me3.bed'), 
                         schema = schema_histone_overlap, 
                         separator="\t",
                         has_header=False)

In [14]:
histone_pl

chromosome,tss -2kb,tss +2kb,test_id,gene_id,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,locus,orig_start,orig_end,histone_chr,histone_start,histone_end,histone_name
str,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,str,i64,i64,str,i64,i64,str
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""chr1""",948266,948412,"""chr1_1429"""
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""chr1""",948408,948554,"""chr1_1430"""
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""chr1""",948573,948719,"""chr1_1431"""
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""chr1""",948913,949059,"""chr1_1432"""
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919,"""chr1""",949098,949244,"""chr1_1433"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",21904825,21908825,"""XLOC_030002""","""XLOC_030002""","""Y""",21865751,21906825,-1,"""KDM5D""","""ENSG00000012817""","""protein_coding""",21906825,"""chrY:21867206-21906914""",21867206,21906914,"""chrY""",21906013,21906159,"""chrY_44470"""
"""chrY""",21904825,21908825,"""XLOC_030002""","""XLOC_030002""","""Y""",21865751,21906825,-1,"""KDM5D""","""ENSG00000012817""","""protein_coding""",21906825,"""chrY:21867206-21906914""",21867206,21906914,"""chrY""",21906218,21906364,"""chrY_44471"""
"""chrY""",21904825,21908825,"""XLOC_030002""","""XLOC_030002""","""Y""",21865751,21906825,-1,"""KDM5D""","""ENSG00000012817""","""protein_coding""",21906825,"""chrY:21867206-21906914""",21867206,21906914,"""chrY""",21906625,21906771,"""chrY_44472"""


In [15]:
# Create an index for the dataframe
histone_partitions = histone_pl.partition_by("gene_id", as_dict=True)

In [18]:
len(histone_partitions)

14014

In [28]:
histone_list = []
i = 1
for gene in gene_list:
    histone_arr = np.zeros(4000, dtype=int)
    count = 0
    
    if((gene,) in histone_partitions):
        # print(f"{gene} -  EXIST")
        
        partition = histone_partitions[(gene,)].to_numpy()
        count = partition.shape[0]

        for row in partition:
            start = row[1]
            idx_start = row[17] - start
            idx_end = row[18] - start
            # print(f"{row[19]}: {idx_start} - {idx_end}")
            
            histone_arr[idx_start: idx_end] = 1
    # else:
    #     print(f"{gene} -  NOT EXIST")

    # print(f"SUM: {np.sum(histone_arr)}, COUNT: {count}")
    entry = [gene, histone_arr, count]

    # print(f"{i}/{len(gene_list)}: {gene}, count: {count}")

    histone_list.append(entry)
    i += 1

In [29]:
len(histone_list)

22154

In [30]:
histone_np = np.array(histone_list, dtype="object")

In [31]:
histone_pd = pd.DataFrame(histone_np, columns=["gene_id", f"{histone_name}", f"{histone_name}_count"])

In [32]:
histone_pd

,gene_id,H3K4me3,H3K4me3_count
0,XLOC_000001,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
1,XLOC_000003,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
2,XLOC_000006,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
3,XLOC_000007,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
4,XLOC_000008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",6
...,...,...,...
22149,XLOC_030009,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
22150,XLOC_030012,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
22151,XLOC_030014,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
22152,XLOC_030017,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0


In [33]:
histone_pl = pl.from_pandas(histone_pd)

In [34]:
histone_pl

gene_id,H3K4me3,H3K4me3_count
str,list[i64],i64
"""XLOC_000001""","[0, 0, … 0]",0
"""XLOC_000003""","[0, 0, … 0]",0
"""XLOC_000006""","[0, 0, … 0]",0
"""XLOC_000007""","[0, 0, … 0]",0
"""XLOC_000008""","[0, 0, … 0]",6
…,…,…
"""XLOC_030009""","[0, 0, … 0]",0
"""XLOC_030012""","[0, 0, … 0]",0
"""XLOC_030014""","[0, 0, … 0]",0
